# 🖼️ CNN-Autoencoder for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **Convolutional Neural Network (CNN) Autoencoder** for anomaly detection in brain MRI scans.

### Model Architecture
- **Type**: Convolutional Autoencoder
- **Input**: 128×128 grayscale MRI slice
- **Encoder**: 4 Conv2D layers with MaxPooling
- **Decoder**: 4 ConvTranspose2D layers for upsampling
- **Latent**: 256-dimensional bottleneck

### Key Features
- **Spatial feature extraction** with convolutional layers
- **Translation equivariance** (but NOT rotation equivariant)
- **Robust checkpoint recovery** for Colab disconnects
- **Scheduler state preserved** on resume
- **Auto-cleanup**: Keeps only last 3 checkpoints (saves Drive space)
- **Data integrity check** before training

### Expected Performance
- **AUROC**: ~0.82-0.87 (better than baseline)
- **Training Time**: ~30-45 min per epoch, ~25-35 hours total

### ⚡ Optimizations
- Batch size 64 for faster training
- 50 epochs for full convergence
- Mixed precision (FP16) for 2-3x speedup
- Checkpoint every epoch with auto-cleanup
- Auto-resume from last checkpoint
- Keep-alive script included

---

## 1️⃣ Setup and Environment Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

print("✅ Google Drive mounted successfully!")

In [ ]:
# Keep Colab session alive (prevents random disconnects)
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

In [ ]:
# Install required packages
!pip install pytorch-msssim -q

print("✅ All packages installed!")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler  # Mixed precision training
from pytorch_msssim import SSIM

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from glob import glob
import os
import time
from tqdm import tqdm
import json

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

print("✅ All libraries imported successfully!")

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"

# Data paths (matches preprocessing output)
IXI_TRAIN_PATH = f"{BASE_PATH}/data/ixi_t1/processed_ixi/train"
IXI_VAL_PATH = f"{BASE_PATH}/data/ixi_t1/processed_ixi/val"
BRATS_PATH = f"{BASE_PATH}/data/brats_t1/test"  # Centered BraTS data

# Separate model folders for CNN-AE and ECNN-AE
MODEL_PATH = f"{BASE_PATH}/models/saved_models/cnn_ae"  # CNN-AE specific folder
RESULTS_PATH = f"{BASE_PATH}/results/cnn_autoencoder"

# Create directories if they don't exist
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("📁 Paths configured:")
print(f"   IXI Train: {IXI_TRAIN_PATH}")
print(f"   IXI Val: {IXI_VAL_PATH}")
print(f"   BraTS Test: {BRATS_PATH}")
print(f"   CNN-AE Models: {MODEL_PATH}")
print(f"   Results: {RESULTS_PATH}")

In [ ]:
# ============================================
# DATA INTEGRITY CHECK - Run BEFORE training
# ============================================
print("🔍 Data Integrity Check...")
print("=" * 60)

# Check if directories exist
for name, path in [("IXI Train", IXI_TRAIN_PATH), ("IXI Val", IXI_VAL_PATH), ("BraTS Test", BRATS_PATH)]:
    if os.path.exists(path):
        files = glob(f"{path}/*.npy")
        print(f"✅ {name}: {len(files):,} files")
    else:
        print(f"❌ {name}: Directory NOT FOUND!")
        print(f"   Expected: {path}")

# Sample file check - verify dimensions
print("-" * 60)
print("📐 Sample File Dimensions:")

sample_paths = [
    (IXI_TRAIN_PATH, "IXI Train"),
    (BRATS_PATH, "BraTS Test")
]

for path, name in sample_paths:
    if os.path.exists(path):
        sample_files = glob(f"{path}/*.npy")[:1]
        if sample_files:
            sample = np.load(sample_files[0])
            print(f"   {name}: {sample.shape} | Range: [{sample.min():.4f}, {sample.max():.4f}]")
            if sample.shape != (128, 128):
                print(f"   ⚠️ WARNING: Expected (128, 128), got {sample.shape}")

print("-" * 60)
print("✅ Data integrity check complete!")

## 2️⃣ Data Loading and Preprocessing

In [ ]:
# Dataset class for MRI slices
class MRIDataset(Dataset):
    """
    Dataset for loading preprocessed MRI slices (.npy files)
    Returns: (image, image) pairs for autoencoder training
    """
    def __init__(self, file_list):
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            # Load .npy file
            img = np.load(self.files[idx])

            # Add channel dimension: (128, 128) -> (1, 128, 128)
            img = np.expand_dims(img, 0)

            # Convert to tensor
            img_tensor = torch.from_numpy(img).float()

            # Return (input, target) pair - same for autoencoder
            return img_tensor, img_tensor
        except Exception as e:
            # Fallback for corrupted files
            print(f"Error loading {self.files[idx]}: {e}")
            return torch.zeros((1, 128, 128)), torch.zeros((1, 128, 128))

print("✅ Dataset class defined!")

In [ ]:
# Load file paths
print("📂 Loading file paths...")

train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

# Validation checks
if len(train_files) == 0:
    raise ValueError(f"❌ No training files found in {IXI_TRAIN_PATH}!")

if len(val_files) == 0:
    raise ValueError(f"❌ No validation files found in {IXI_VAL_PATH}!")

if len(brats_files) == 0:
    print(f"⚠️ WARNING: No BraTS test files found in {BRATS_PATH}")
    print("   Make sure BraTS centering post-processing is complete!")

print(f"✅ Found {len(train_files):,} IXI training slices")
print(f"✅ Found {len(val_files):,} IXI validation slices")
print(f"✅ Found {len(brats_files):,} BraTS test slices (tumors)")

In [ ]:
# Create datasets and dataloaders
train_dataset = MRIDataset(train_files)
val_dataset = MRIDataset(val_files)
test_dataset = MRIDataset(brats_files)

# Optimized batch size for CNN (larger than baseline for speed)
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 3️⃣ CNN-Autoencoder Model Architecture

In [ ]:
class CNNAutoencoder(nn.Module):
    """
    Convolutional Autoencoder

    Encoder: 128×128 -> 64×64 -> 32×32 -> 16×16 -> 8×8
    Decoder: 8×8 -> 16×16 -> 32×32 -> 64×64 -> 128×128
    """

    def __init__(self, latent_dim=256):
        super(CNNAutoencoder, self).__init__()

        # Encoder: Progressive downsampling
        self.encoder = nn.Sequential(
            # 128×128×1 -> 128×128×32
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 64×64

            # 64×64×32 -> 64×64×64
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 32×32

            # 32×32×64 -> 32×32×128
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 16×16

            # 16×16×128 -> 16×16×256
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2)  # 8×8
        )

        # Latent space
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)

        # Decoder: Progressive upsampling
        self.decoder = nn.Sequential(
            # 8×8×256 -> 16×16×256
            nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # 16×16×256 -> 32×32×128
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 32×32×128 -> 64×64×64
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # 64×64×64 -> 128×128×32
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            # 128×128×32 -> 128×128×1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encode
        features = self.encoder(x)

        # Latent
        batch_size = features.size(0)
        flat = self.flatten(features)
        z = self.fc_encode(flat)

        # Decode
        decoded_flat = self.fc_decode(z)
        decoded_features = decoded_flat.view(batch_size, 256, 8, 8)
        x_recon = self.decoder(decoded_features)

        return x_recon

    def get_latent(self, x):
        features = self.encoder(x)
        flat = self.flatten(features)
        return self.fc_encode(flat)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Added this line
model = CNNAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())

print("🖼️ CNN-Autoencoder Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.2f} MB")

## 4️⃣ Loss Function and Optimizer

In [ ]:
class CombinedLoss(nn.Module):
    """
    Combined MSE + SSIM Loss (using SSIM for 128×128 images)

    Loss = α * MSE + (1-α) * (1 - SSIM)
    """
    def __init__(self, alpha=0.84):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ssim = SSIM(data_range=1.0, size_average=True, channel=1, win_size=11)

    def forward(self, output, target):
        mse_loss = self.mse(output, target)
        ssim_loss = 1 - self.ssim(output, target)
        combined = self.alpha * mse_loss + (1 - self.alpha) * ssim_loss
        return combined

criterion = CombinedLoss(alpha=0.84)

print("✅ Combined Loss Function created!")
print(f"   MSE weight (α): 0.84")
print(f"   SSIM weight (1-α): 0.16")

In [ ]:
def create_optimizer_scheduler(model):
    """Create optimizer and scheduler"""
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    return opt, sched

# Create GradScaler for mixed precision training

scaler = GradScaler()
print("✅ Optimizer factory and GradScaler defined!")


## 5️⃣ Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, scaler):
    """Train with mixed precision (FP16) for 2-3x speedup"""
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc='Training')
    for data, target in pbar:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast(): # Changed from autocast('cuda') to autocast()
            output = model(data)
            loss = criterion(output, target)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    return running_loss / len(dataloader)

In [ ]:
# Delete broken checkpoints
!rm {MODEL_PATH}/cnn_*.pth
print("✅ Checkpoints deleted, ready for clean restart")

## 6️⃣ Checkpoint Recovery

In [ ]:
# ============================================
# CHECKPOINT RECOVERY with Scheduler State
# ============================================
checkpoints = sorted(glob(f'{MODEL_PATH}/cnn_epoch*.pth'))
RESUME = len(checkpoints) > 0

if RESUME:
    latest = checkpoints[-1]
    print(f"✅ Found checkpoint: {os.path.basename(latest)}")
    checkpoint = torch.load(latest, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer, scheduler = create_optimizer_scheduler(model)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Restore scheduler state if available
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("   Scheduler state restored!")
    
    start_epoch = checkpoint['epoch']
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    best_epoch = checkpoint.get('best_epoch', 0)
    print(f"   Resuming from epoch {start_epoch}")
    print(f"   Current LR: {optimizer.param_groups[0]['lr']:.2e}")
else:
    optimizer, scheduler = create_optimizer_scheduler(model)
    start_epoch = 0
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_epoch = 0
    print("📝 Starting fresh training")

## 7️⃣ Main Training Loop

In [ ]:
NUM_EPOCHS = 50  # Full training: 50 epochs for best results
KEEP_LAST_N_CHECKPOINTS = 3  # Cleanup: keep only last 3 checkpoints

print(f"🚀 Training CNN-AE: Epochs {start_epoch + 1} to {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}, Device: {device}")
print(f"   Keeping last {KEEP_LAST_N_CHECKPOINTS} checkpoints")
print("-" * 60)

start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {current_lr:.2e} | Time: {epoch_time/60:.1f}min")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),  # Save scheduler state
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch
        }, f'{MODEL_PATH}/cnn_best.pth')
        print(f"   ✅ Best model saved!")

    # Save checkpoint every epoch (with scheduler state)
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),  # Save scheduler state
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch
    }, f'{MODEL_PATH}/cnn_epoch{epoch+1}.pth')
    
    # Cleanup: Keep only last N checkpoints (saves Drive space)
    checkpoints = sorted(glob(f'{MODEL_PATH}/cnn_epoch*.pth'))
    if len(checkpoints) > KEEP_LAST_N_CHECKPOINTS:
        for old_ckpt in checkpoints[:-KEEP_LAST_N_CHECKPOINTS]:
            os.remove(old_ckpt)
            print(f"   🗑️ Deleted old checkpoint: {os.path.basename(old_ckpt)}")

    print("-" * 60)

total_time = time.time() - start_time
print(f"\n🎉 Training Complete!")
print(f"   Total Time: {total_time/3600:.2f} hours")
print(f"   Best Epoch: {best_epoch}, Best Val Loss: {best_val_loss:.6f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.axvline(x=best_epoch-1, color='r', linestyle='--', label=f'Best ({best_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN-AE Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train', linewidth=2)
plt.plot(val_losses, label='Val', linewidth=2)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log)')
plt.title('CNN-AE Training (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_training_curves.png', dpi=150)
plt.show()

## 8️⃣ Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(f'{MODEL_PATH}/cnn_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Best model loaded (Epoch {checkpoint['epoch']}, Val Loss: {checkpoint['val_loss']:.6f})")

# Calculate reconstruction errors
def calculate_errors(model, dataloader, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Computing errors'):
            data = data.to(device)
            recon = model(data)
            mse = nn.functional.mse_loss(recon, data, reduction='none').view(data.size(0), -1).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

normal_errors = calculate_errors(model, val_loader, device)
anomaly_errors = calculate_errors(model, test_loader, device)

# Metrics
y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])
auroc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print(f"\n📈 CNN-AE Performance:")
print(f"   AUROC: {auroc:.4f}")
print(f"   AUPRC: {auprc:.4f}")
print(f"   Normal errors: {normal_errors.mean():.6f} ± {normal_errors.std():.6f}")
print(f"   Anomaly errors: {anomaly_errors.mean():.6f} ± {anomaly_errors.std():.6f}")

# Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal', density=True, color='blue')
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly', density=True, color='red')
plt.xlabel('Reconstruction Error')
plt.ylabel('Density')
plt.legend()
plt.title('CNN-AE: Error Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, linewidth=2, label=f'CNN-AE (AUROC={auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('CNN-AE: ROC Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_evaluation.png', dpi=150)
plt.show()

# Save results
results = {
    'model': 'CNN-Autoencoder',
    'auroc': float(auroc),
    'auprc': float(auprc),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'normal_error_mean': float(normal_errors.mean()),
    'anomaly_error_mean': float(anomaly_errors.mean()),
    'total_params': total_params,
    'training_time_hours': total_time / 3600
}

with open(f'{RESULTS_PATH}/cnn_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\n✅ CNN-AE evaluation complete!")

## 9️⃣ Visualization: Reconstruction Maps & Anomaly Localization


In [ ]:
def visualize_reconstructions(model, dataloader, device, num_samples=5, title_prefix=""):
    """
    Visualize original images, reconstructions, and error maps

    Args:
        model: Trained autoencoder
        dataloader: DataLoader to sample from
        device: torch device
        num_samples: Number of samples to visualize
        title_prefix: Prefix for plot title (e.g., "Normal" or "Anomaly")
    """
    model.eval()

    # Get a batch of images
    data, _ = next(iter(dataloader))
    data = data[:num_samples].to(device)

    # Generate reconstructions
    with torch.no_grad():
        recon = model(data)

    # Calculate error maps (absolute difference)
    error_maps = torch.abs(data - recon)

    # Move to CPU for visualization
    data_np = data.cpu().numpy()
    recon_np = recon.cpu().numpy()
    error_np = error_maps.cpu().numpy()

    # Create figure
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_samples):
        # Original
        axes[i, 0].imshow(data_np[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[i, 0].set_title(f'{title_prefix} Original', fontsize=12, fontweight='bold')
        axes[i, 0].axis('off')

        # Reconstruction
        axes[i, 1].imshow(recon_np[i, 0], cmap='gray', vmin=0, vmax=1)
        mse_val = np.mean((data_np[i, 0] - recon_np[i, 0])**2)
        axes[i, 1].set_title(f'Reconstruction (MSE={mse_val:.6f})', fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')

        # Error map (difference)
        im = axes[i, 2].imshow(error_np[i, 0], cmap='hot', vmin=0, vmax=error_np[i, 0].max())
        axes[i, 2].set_title(f'Error Map (Max={error_np[i, 0].max():.4f})', fontsize=12, fontweight='bold')
        axes[i, 2].axis('off')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046)

    plt.tight_layout()
    return fig

print("✅ Visualization function defined!")

# Visualize Normal Cases (IXI Validation - Healthy Brains)
print("🔍 Visualizing Normal Cases (Healthy Brains)...")
fig_normal = visualize_reconstructions(model, val_loader, device, num_samples=5, title_prefix="Normal")
plt.savefig(f'{RESULTS_PATH}/cnn_reconstruction_normal.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Normal reconstructions saved!")
# Visualize Anomaly Cases (BraTS - Brain Tumors)
print("🔍 Visualizing Anomaly Cases (Brain Tumors)...")
fig_anomaly = visualize_reconstructions(model, test_loader, device, num_samples=5, title_prefix="Anomaly")
plt.savefig(f'{RESULTS_PATH}/cnn_reconstruction_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Anomaly reconstructions saved!")

print("\n" + "="*60)
print("📊 INTERPRETATION:")
print("="*60)
print("🔵 NORMAL (Healthy): Low MSE, minimal red in error maps")
print("   → Model learned healthy brain patterns well")
print("🔴 ANOMALY (Tumors): High MSE, bright red regions in error maps")
print("   → Tumor regions have high reconstruction error")
print("   → Red areas indicate WHERE the anomaly is located")
print("="*60)